# Benchmark validation: ClinVar-conflicting variants + the MYH7 gold-standard set

Runs the pipeline on ClinVar variants classified **"Conflicting classifications of pathogenicity"** -- i.e. variants ClinVar's own submitters could not agree on -- and checks what CardioClassifier calls them. Unlike a concordance check (there's no agreed "truth" to compare against for a conflicting variant by definition), the question here is: for variants humans disagreed on, does the pipeline's own evidence-based call land somewhere clearly interpretable (P/LP or B/LB), or does it mostly land on VUS too?

Self-contained (doesn't `%run initial_validationn.ipynb`, since that would also re-trigger its own ~100-variant "clear" batch as a side effect) -- duplicates the small set of helper functions it needs from there.

Patient-specific fields (de novo, zygosity, phasing, segregation, PP4, BS2, BP5) are set to **honest "unknown"** defaults, not a fabricated negative "n" -- there is no real patient behind a retrospective ClinVar record, and asserting e.g. "not de novo" would be a false claim, not a neutral one. `zygosity` is the one exception (defaults to "het"): the pipeline only has het/hom, no "unknown" state, since it feeds a real BA1/BS1 threshold adjustment for biallelic genes.


In [ ]:
# --------------------------------------------------
# Load g2p_clean / disease_ref / classify_variant etc. directly
# --------------------------------------------------
if "g2p_clean" not in globals():
    %run annotationn.ipynb
    %run classifierr.ipynb


In [ ]:
# --------------------------------------------------
# ClinVar search helper (NCBI eutils) -- same logic as find_conflicting_variants
# in initial_validationn.ipynb, duplicated here so this notebook doesn't depend on it.
# --------------------------------------------------
import requests
import re
import time

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def _gene_term(genes):
    return "(" + " OR ".join(f"{g}[gene]" for g in genes) + ")"


_NCBI_TRANSIENT_STATUS = {429, 500, 502, 503, 504}


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    """
    Retries on 429 (rate limit) and 5xx with exponential backoff (2s, 4s,
    8s...). Unauthenticated eutils access is capped at 3 req/sec, but even a
    correctly-paced sequence of many batched calls can still trip a 429
    partway through a large pull. A 429/5xx here means "wait and retry",
    not "the request is malformed" -- a genuine 4xx is NOT retried.
    """
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code in _NCBI_TRANSIENT_STATUS and attempt < max_retries:
            wait = backoff_base ** attempt
            print(f"  NCBI HTTP {r.status_code}; retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()
    return r


def _esearch(term, retmax):
    r = _ncbi_get(f"{EUTILS}/esearch.fcgi",
                  params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return r.json()["esearchresult"]["idlist"]


def _esummary(ids, batch_size=45, pause=0.34):
    """Batched, paced esummary lookup -- stays comfortably under NCBI's unauthenticated 3 req/sec limit."""
    out = {}
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        r = _ncbi_get(f"{EUTILS}/esummary.fcgi",
                      params={"db": "clinvar", "id": ",".join(batch), "retmode": "json"})
        res = r.json()["result"]
        out.update({uid: doc for uid, doc in res.items() if uid != "uids"})
        time.sleep(pause)
    return out


def _extract_cdna_hgvs(title, gene):
    """'NM_000256.3(MYBPC3):c.405A>G (p.Lys135=)' -> 'NM_000256.3(MYBPC3):c.405A>G'"""
    m = re.match(r"^(NM_\d+\.\d+)\([A-Z0-9]+\):(c\.[^\s]+)", title or "")
    return f"{m.group(1)}({gene}):{m.group(2)}" if m else None


def find_conflicting_variants(genes, retmax=600):
    """ClinVar variants explicitly classified 'Conflicting classifications of pathogenicity'."""
    term = f'{_gene_term(genes)} AND "conflicting classifications of pathogenicity"[Clinical_significance]'
    summaries = _esummary(_esearch(term, retmax))

    hits = []
    for uid, doc in summaries.items():
        gc = doc.get("germline_classification", {}) or {}
        if "conflicting" not in str(gc.get("description") or "").lower():
            continue
        genes_field = doc.get("genes") or []
        gene = genes_field[0].get("symbol") if genes_field else None
        hgvs = _extract_cdna_hgvs(doc.get("title"), gene) if gene else None
        if gene and hgvs:
            hits.append({"uid": uid, "gene": gene, "hgvs": hgvs,
                         "clinvar_significance": gc.get("description"),
                         "clinvar_review_status": gc.get("review_status")})
    return hits


In [ ]:
# --------------------------------------------------
# Auto-inferring disease_short_code from a ClinVar-reported gene
# (same logic as initial_validationn.ipynb's infer_disease_short_code)
# --------------------------------------------------
_VALIDITY_RANK = {"definitive": 4, "strong": 3, "moderate": 2, "limited": 1}


def infer_disease_short_code(gene_symbol, g2p_df=None, disease_ref_df=None):
    """Returns (disease_short_code_or_None, is_ambiguous, candidate_codes)."""
    g2p_df = g2p_df if g2p_df is not None else g2p_clean
    disease_ref_df = disease_ref_df if disease_ref_df is not None else disease_ref

    rows = g2p_df[g2p_df["gene_symbol"].astype(str).str.upper() == str(gene_symbol).upper()]
    if rows.empty:
        return None, False, []

    candidates = []
    for _, row in rows.iterrows():
        match = disease_ref_df[disease_ref_df["referral_indication"] == row["referral_indication"]]
        if len(match) > 1:  # HCM split into HCM-FAM/HCM-SYN -- use this row's own subtype to pick
            match = match[match["hcm_subtype"] == row.get("hcm_subtype")]
        if match.empty:
            continue
        rank = _VALIDITY_RANK.get(str(row.get("gene_disease_validity", "")).strip().lower(), 0)
        candidates.append((match.iloc[0]["dis_name"], rank))

    if not candidates:
        return None, False, []

    unique_codes = sorted({code for code, _rank in candidates})
    is_ambiguous = len(unique_codes) > 1
    best_code = sorted(candidates, key=lambda c: (-c[1], c[0]))[0][0]
    return best_code, is_ambiguous, unique_codes


In [ ]:
# --------------------------------------------------
# Pipeline runner -- same logic as initial_validationn.ipynb's run_variant_through_main1
# --------------------------------------------------
# Patient-specific fields default to honest "unknown", not a fabricated
# negative "n" -- there is no real patient behind a retrospective ClinVar
# record. "unknown" and "n" fire identically for every one of these codes
# (apply_ps2_pm6_rule / _manual_curator_hits in classifierr.ipynb only ever
# fire on an explicit affirmative), so this changes what gets truthfully
# recorded, not what fires. zygosity stays "het" -- the pipeline only has
# het/hom, no "unknown" state, since it feeds a real BA1/BS1 adjustment.
import json


def run_variant_through_main1(hgvs, disease_short_code=None, **extra_answers):
    global _VALIDATION_ANSWERS
    _VALIDATION_ANSWERS = {
        "variant": hgvs,
        "disease_short_code": disease_short_code,
        "de_novo_answer": "unknown",
        "zygosity_answer": "het",
        "phasing_answer": "unknown",
        "segregation_answer": "unknown",
        "pp4_answer": "unknown",
        "bs2_answer": "unknown",
        "bp5_answer": "unknown",
        **extra_answers,
    }

    get_ipython().run_line_magic("run", "-i mainn.ipynb")

    gdp = context["gene_disease_pair"]
    return {
        "hgvs": hgvs,
        "gene": context.get("gene_symbol"),
        "pipeline_classification": result["classification"],
        "posterior_probability": result["posterior_probability"],
        "tavtigian_score": result["combined_score"],
        "evidence_codes": result["evidence_codes"],
        "gene_disease_pair_confirmed": gdp["pair_found"],
        "gene_disease_pair_reason": gdp.get("reason"),
    }


In [ ]:
import pandas as pd

ALL_PANEL_GENES = sorted(g2p_clean["gene_symbol"].dropna().unique().tolist())
print(f"{len(ALL_PANEL_GENES)} genes in the panel:", ALL_PANEL_GENES)


In [ ]:
# --------------------------------------------------
# Build the 100-conflicting-variant pool
# --------------------------------------------------
# 14,588 conflicting-classification variants are available across this gene
# panel (confirmed live), so retmax=600 comfortably covers the target
# even after some records get dropped client-side for missing gene/HGVS
# fields.
TARGET_TOTAL = 100

conflicting_pool = find_conflicting_variants(ALL_PANEL_GENES, retmax=600)
print(f"Raw pool: {len(conflicting_pool)} conflicting variants found")

batch_variants = conflicting_pool[:TARGET_TOTAL]
print(f"Batch selected: {len(batch_variants)} variants")


In [ ]:
# --------------------------------------------------
# Resumable batch runner
# --------------------------------------------------
# Same resumable-checkpoint pattern as initial_validationn.ipynb, but its own
# checkpoint file (initial_validationn.ipynb's "clear"/graded batch is a separate,
# unrelated run and shouldn't be touched by this one). Re-running this cell
# skips any hgvs already SUCCESSFULLY recorded; a previously failed variant
# is retried (annotationn.ipynb's VEP call has its own retry/backoff for
# transient 500/502/503/504/timeout errors).
import time
from pathlib import Path

CHECKPOINT_PATH = Path("outputs/sandbox_conflicting_batch.jsonl")
PAUSE_BETWEEN_VARIANTS = 0.5
CHECKPOINT_PATH.parent.mkdir(exist_ok=True)

succeeded_before = set()
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "error" not in rec:
                succeeded_before.add(rec["hgvs"])
    print(f"Resuming: {len(succeeded_before)} variants already succeeded in {CHECKPOINT_PATH} "
          f"(previously-failed variants will be retried)")

n_ok, n_failed = 0, 0
with CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt:
    for i, rec in enumerate(batch_variants, start=1):
        hgvs = rec["hgvs"]
        if hgvs in succeeded_before:
            continue

        gene = rec["gene"]
        disease_code, is_ambiguous, candidates = infer_disease_short_code(gene)

        try:
            result_row = run_variant_through_main1(hgvs, disease_code)
        except Exception as exc:
            n_failed += 1
            ckpt.write(json.dumps({
                "hgvs": hgvs, "gene": gene, "error": str(exc),
                "clinvar_significance": rec["clinvar_significance"],
            }) + "\n")
            ckpt.flush()
            print(f"[{i}/{len(batch_variants)}] FAILED {hgvs}: {exc}")
            time.sleep(PAUSE_BETWEEN_VARIANTS)
            continue

        result_row.update({
            "clinvar_significance":          rec["clinvar_significance"],
            "clinvar_review_status":         rec.get("clinvar_review_status"),
            "disease_short_code_used":       disease_code,
            "disease_short_code_ambiguous":  is_ambiguous,
            "disease_short_code_candidates": candidates,
        })
        ckpt.write(json.dumps(result_row, default=str) + "\n")
        ckpt.flush()
        n_ok += 1

        if i % 25 == 0 or i == len(batch_variants):
            print(f"[{i}/{len(batch_variants)}] {n_ok} ok, {n_failed} failed so far")

        time.sleep(PAUSE_BETWEEN_VARIANTS)

print(f"Done. {n_ok} succeeded, {n_failed} failed this run. Full results in {CHECKPOINT_PATH}")


In [ ]:
# --------------------------------------------------
# Load the checkpoint file and save a clean summary for viewing/reuse
# --------------------------------------------------
_records = []
with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            _records.append(json.loads(line))

sandbox_results_df = pd.DataFrame(_records)

if "error" in sandbox_results_df.columns:
    failed_df = sandbox_results_df[sandbox_results_df["error"].notna()]
    sandbox_results_df = sandbox_results_df[sandbox_results_df["error"].isna()].drop(columns=["error"])
else:
    failed_df = sandbox_results_df.iloc[0:0]

print(f"{len(sandbox_results_df)} variants classified successfully, {len(failed_df)} failed")

_summary_cols = ["hgvs", "gene", "clinvar_significance", "pipeline_classification",
                  "posterior_probability", "tavtigian_score", "evidence_codes",
                  "disease_short_code_used", "disease_short_code_ambiguous"]

sandbox_results_df[_summary_cols].to_csv("outputs/sandbox_conflicting_summary.csv", index=False)
sandbox_results_df.to_json("outputs/sandbox_conflicting_summary.json", orient="records", indent=2)
print("Saved: outputs/sandbox_conflicting_summary.csv, outputs/sandbox_conflicting_summary.json")

sandbox_results_df[_summary_cols].head(20)


## Where do conflicting variants land once the pipeline actually evaluates them?

There's no single "truth" tier to score these against (that's the whole point of "conflicting"), so this isn't a concordance plot -- it's a distribution: of variants humans couldn't agree on, how many does the pipeline's own evidence resolve to something actionable (Pathogenic/Likely Pathogenic or Benign/Likely Benign) versus how many still land on VUS.


In [ ]:
import matplotlib.pyplot as plt

_TIER_ORDER = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]
# Ordinal blue ramp (light->dark encodes Benign->Pathogenic), same sequential
# hue family as the concordance plot in initial_validationn.ipynb, steps chosen from
# the documented ordinal range (lightest clears 2:1 contrast on a light surface).
_TIER_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]

counts = sandbox_results_df["pipeline_classification"].value_counts().reindex(_TIER_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.bar(_TIER_ORDER, counts.values, color=_TIER_COLORS, width=0.6)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.015,
            str(count), ha="center", va="bottom", fontsize=10, color="#0b0b0b")

n = len(sandbox_results_df)
actionable = counts[["Pathogenic", "Likely Pathogenic", "Benign", "Likely Benign"]].sum()
ax.set_title(f"Pipeline classification of {n} ClinVar-conflicting variants\n"
             f"{actionable}/{n} ({actionable/n:.0%}) resolved to Pathogenic/Likely Pathogenic/Benign/Likely Benign",
             fontsize=11, color="#0b0b0b")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_ylim(0, max(counts.values) * 1.15)

ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")

fig.tight_layout()
fig.savefig("outputs/sandbox_conflicting_distribution.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(counts.to_string())


## What did the individual ClinVar submitters actually say?

`esummary` (used everywhere above) only exposes ClinVar's *aggregate* call -- for every variant in this batch that's just the label "Conflicting classifications of pathogenicity", with no indication of what the underlying disagreement actually was. The real per-submitter breakdown (which lab said what) requires a different call: `esearch` to resolve the HGVS to a ClinVar variation ID, then `efetch` with `rettype=vcv`, which returns the full record including one `<ClinicalAssertion>` per submission.

This is independent of `annotationn.ipynb`/`classifierr.ipynb`/`mainn.ipynb` -- no VEP, no gnomAD, no classification logic -- so it reuses the pipeline classifications already computed above rather than re-running them.


In [ ]:
# --------------------------------------------------
# Standalone re-entry point for this section
# --------------------------------------------------
# Cells 12-14 below only need the ClinVar esearch/esummary helpers and the
# finished 100-variant results table -- not VEP, gnomAD, or classification
# logic (see the markdown above). Re-running the full batch (cells 1-8)
# just to get here would be wasteful and slow, so this reloads the already-
# saved summary from disk instead. Guarded with `not in globals()`, same
# pattern as cell 1, so running the whole notebook top-to-bottom still
# works unchanged -- this is a no-op when sandbox_results_df already exists.
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
_NCBI_TRANSIENT_STATUS = {429, 500, 502, 503, 504}


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    """Retries on 429/5xx with exponential backoff; a genuine 4xx is not retried."""
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code in _NCBI_TRANSIENT_STATUS and attempt < max_retries:
            wait = backoff_base ** attempt
            print(f"  NCBI HTTP {r.status_code}; retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()
    return r


def _esearch(term, retmax):
    r = _ncbi_get(f"{EUTILS}/esearch.fcgi",
                  params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return r.json()["esearchresult"]["idlist"]


if "sandbox_results_df" not in globals():
    sandbox_results_df = pd.read_json("outputs/sandbox_conflicting_summary.json")
    print(f"Reloaded {len(sandbox_results_df)} variants from outputs/sandbox_conflicting_summary.json")


In [ ]:
# --------------------------------------------------
# Per-submitter ClinVar classifications
# --------------------------------------------------
# Confirmed live against real variants, e.g. NM_004415.4(DSP):c.4713G>T --
# Labcorp/Invitae called it "Uncertain significance", Ambry Genetics called
# it "Likely benign"; esummary only ever showed "Conflicting classifications
# of pathogenicity" for this variant, with no way to see that breakdown.
def _parse_clinical_assertions(vcv_xml):
    """Extract (submitter, classification, review_status) for every <ClinicalAssertion> in a ClinVar VCV efetch response."""
    assertions = []
    for block in re.findall(r"<ClinicalAssertion .*?</ClinicalAssertion>", vcv_xml, re.S):
        submitter_m = re.search(r'SubmitterName="([^"]*)"', block)
        classification_m = re.search(r"<GermlineClassification>([^<]*)</GermlineClassification>", block)
        review_m = re.search(r"<ReviewStatus>([^<]*)</ReviewStatus>", block)
        if submitter_m and classification_m:
            assertions.append({
                "submitter": submitter_m.group(1),
                "classification": classification_m.group(1),
                "review_status": review_m.group(1) if review_m else None,
            })
    return assertions


def fetch_submitter_classifications(hgvs):
    """
    Returns a list of {submitter, classification, review_status} dicts --
    one per ClinVar submission contributing to this variant's aggregate
    call. Returns [] if the variant can't be resolved to a ClinVar UID.
    """
    ids = _esearch(hgvs, retmax=1)
    if not ids:
        return []
    r = _ncbi_get(f"{EUTILS}/efetch.fcgi",
                  params={"db": "clinvar", "id": ids[0], "rettype": "vcv",
                          "is_variationid": "true", "retmode": "xml"})
    return _parse_clinical_assertions(r.text)


In [ ]:
# --------------------------------------------------
# Resumable per-submitter fetch for the batch's HGVS list
# --------------------------------------------------
# Reuses the HGVS list from the already-completed classification run above
# -- does NOT touch or re-run annotation/classification. Own checkpoint
# file, same resumable pattern as the classification batch runner: 2 NCBI
# calls per variant (esearch + efetch), so ~200 calls total for 100
# variants -- lighter than the full classification pipeline (no VEP,
# gnomAD, or PS1/PM5 lookups), but still paced and retried the same way.
SUBMITTER_CHECKPOINT_PATH = Path("outputs/sandbox_conflicting_submitters.jsonl")
PAUSE_BETWEEN_FETCHES = 0.5
SUBMITTER_CHECKPOINT_PATH.parent.mkdir(exist_ok=True)

_hgvs_list = sandbox_results_df["hgvs"].tolist()

submitter_done = set()
if SUBMITTER_CHECKPOINT_PATH.exists():
    with SUBMITTER_CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                if "error" not in rec:
                    submitter_done.add(rec["hgvs"])
    print(f"Resuming: {len(submitter_done)} variants already fetched in {SUBMITTER_CHECKPOINT_PATH}")

n_ok, n_failed = 0, 0
with SUBMITTER_CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt:
    for i, hgvs in enumerate(_hgvs_list, start=1):
        if hgvs in submitter_done:
            continue

        try:
            submitters = fetch_submitter_classifications(hgvs)
        except Exception as exc:
            n_failed += 1
            ckpt.write(json.dumps({"hgvs": hgvs, "error": str(exc)}) + "\n")
            ckpt.flush()
            print(f"[{i}/{len(_hgvs_list)}] FAILED {hgvs}: {exc}")
            time.sleep(PAUSE_BETWEEN_FETCHES)
            continue

        ckpt.write(json.dumps({"hgvs": hgvs, "submitters": submitters}) + "\n")
        ckpt.flush()
        n_ok += 1

        if i % 50 == 0 or i == len(_hgvs_list):
            print(f"[{i}/{len(_hgvs_list)}] {n_ok} ok, {n_failed} failed so far")

        time.sleep(PAUSE_BETWEEN_FETCHES)

print(f"Done. {n_ok} succeeded, {n_failed} failed this run. Full results in {SUBMITTER_CHECKPOINT_PATH}")


In [ ]:
# --------------------------------------------------
# Merge submitter data onto the pipeline results
# --------------------------------------------------
# Both sides mapped onto the same 1-5 ordinal scale (not string-matched
# directly) since ClinVar submitters write "Uncertain significance" while
# classify_variant() returns "VUS" for the same tier.
_ORDINAL_TIER = {
    "benign": 1, "likely benign": 2,
    "uncertain significance": 3, "vus": 3,
    "likely pathogenic": 4, "pathogenic": 5,
}


def _tier(label):
    return _ORDINAL_TIER.get(str(label).strip().lower())


_submitter_records = []
with SUBMITTER_CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            _submitter_records.append(json.loads(line))

_submitters_by_hgvs = {r["hgvs"]: r.get("submitters", []) for r in _submitter_records if "error" not in r}

combined_rows = []
for row in sandbox_results_df.to_dict("records"):
    hgvs = row["hgvs"]
    submitters = _submitters_by_hgvs.get(hgvs, [])
    submitter_tiers = [t for t in (_tier(s["classification"]) for s in submitters) if t is not None]
    pipeline_tier = _tier(row["pipeline_classification"])

    row = dict(row)
    row["submitters"] = submitters
    row["submitter_classifications"] = [s["classification"] for s in submitters]
    row["matches_any_submitter"] = (pipeline_tier in submitter_tiers) if submitter_tiers else None
    row["within_submitter_range"] = (
        min(submitter_tiers) <= pipeline_tier <= max(submitter_tiers)
        if submitter_tiers and pipeline_tier is not None else None
    )
    combined_rows.append(row)

combined_df = pd.DataFrame(combined_rows)
combined_df.to_json("outputs/sandbox_conflicting_with_submitters.json", orient="records", indent=2)

# CSV companion -- list/dict columns (evidence_codes, submitters,
# submitter_classifications) don't serialize cleanly as bare CSV cells, so
# those are JSON-stringified for a readable single cell rather than dropped.
_csv_df = combined_df.copy()
for _col in ["evidence_codes", "submitters", "submitter_classifications",
             "disease_short_code_candidates"]:
    if _col in _csv_df.columns:
        _csv_df[_col] = _csv_df[_col].apply(json.dumps)
_csv_df.to_csv("outputs/sandbox_conflicting_with_submitters.csv", index=False)

_has_submitters = combined_df["submitters"].map(len).gt(0)
_match_valid = combined_df["matches_any_submitter"].dropna().astype(bool)
_range_valid = combined_df["within_submitter_range"].dropna().astype(bool)

print(f"{_has_submitters.sum()}/{len(combined_df)} variants had resolvable submitter data")
print(f"Matches at least one submitter: {_match_valid.mean():.1%} (n={len(_match_valid)})")
print(f"Falls within submitted range:   {_range_valid.mean():.1%} (n={len(_range_valid)})")
print("Saved: outputs/sandbox_conflicting_with_submitters.json, outputs/sandbox_conflicting_with_submitters.csv")

combined_df[["hgvs", "gene", "pipeline_classification", "submitter_classifications",
             "matches_any_submitter", "within_submitter_range"]].head(20)


In [ ]:
# --------------------------------------------------
# Pipeline vs. submitter agreement -- three-way breakdown
# --------------------------------------------------
# Ordinal agreement scale (outside range < within range < exact match), so
# one hue light->dark rather than three unrelated categorical colors --
# darker = closer agreement with what ClinVar's submitters actually said.
# Steps drawn from the validated sequential blue ramp (dataviz skill
# reference palette), kept in the ramp's lighter half for a soft, pastel
# feel while still clearing the ordinal 2:1 contrast floor.
_agreement_base = combined_df[combined_df["submitters"].map(len) > 0].copy()


def _agreement_bucket(row):
    if row["matches_any_submitter"]:
        return "Matches a submitter"
    if row["within_submitter_range"]:
        return "Within submitted range"
    return "Outside submitted range"


_agreement_base["agreement_bucket"] = _agreement_base.apply(_agreement_bucket, axis=1)

_BUCKET_ORDER = ["Outside submitted range", "Within submitted range", "Matches a submitter"]
_BUCKET_COLORS = ["#86b6ef", "#6da7ec", "#5598e7"]

_counts = _agreement_base["agreement_bucket"].value_counts().reindex(_BUCKET_ORDER, fill_value=0)
_n = len(_agreement_base)

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.barh(_BUCKET_ORDER, _counts.values, color=_BUCKET_COLORS, height=0.55, zorder=3)
_xmax = max(_counts.values) if len(_counts) and max(_counts.values) > 0 else 1
for bar, count in zip(bars, _counts.values):
    pct = count / _n if _n else 0
    ax.text(bar.get_width() + _xmax * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{count}  ({pct:.0%})", va="center", ha="left", fontsize=10, color="#0b0b0b")

ax.set_title(f"Does the pipeline's call agree with what ClinVar submitters actually said?\n"
             f"n={_n} conflicting variants with resolvable per-submitter data",
             fontsize=11, color="#0b0b0b")
ax.set_xlabel("Number of variants", color="#52514e")
ax.set_xlim(0, _xmax * 1.25)

ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")

fig.tight_layout()
fig.savefig("outputs/sandbox_submitter_agreement.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(_counts.to_string())
print(f"(excluded: {len(combined_df) - _n} variants with no resolvable submitter data)")


In [ ]:
# --------------------------------------------------
# The 8% -- variants where the pipeline's call falls entirely outside
# what any ClinVar submitter said
# --------------------------------------------------
outside_range_df = _agreement_base[_agreement_base["agreement_bucket"] == "Outside submitted range"][
    ["hgvs", "gene", "pipeline_classification", "clinvar_significance",
     "submitter_classifications", "posterior_probability", "evidence_codes"]
].reset_index(drop=True)

print(f"{len(outside_range_df)} variants outside the submitted range")
outside_range_df.head(10)


## Which ACMG codes are actually firing for these conflicting variants?

Same split as `initial_validationn.ipynb`'s equivalent figure: **case-level** codes (PM3, PP1, PP4, BS2, BS4, BP2, BP5, PS2, PM6 -- need patient/family information no retrospective ClinVar pull can supply) versus **computational** (the other 19). For a batch that's specifically the *hard* variants -- the ones ClinVar's own submitters couldn't agree on -- this is exactly where you'd most want case-level evidence (segregation, de novo status) to help resolve the disagreement, and exactly where this retrospective validation structurally cannot supply it.


In [ ]:
from collections import Counter
import numpy as np

CASE_LEVEL_CODES = {"PM3", "PP1", "PP4", "BS2", "BS4", "BP2", "BP5", "PS2", "PM6"}
ALL_ACMG_CODES = ["PVS1", "PS1", "PS2", "PS3", "PS4", "PM1", "PM2", "PM3", "PM4", "PM5", "PM6",
                   "PP1", "PP2", "PP3", "PP4", "PP5", "BA1", "BS1", "BS2", "BS3", "BS4",
                   "BP1", "BP2", "BP3", "BP4", "BP5", "BP6", "BP7"]


def rule_activation_frequencies(df):
    n = len(df)
    counts = Counter()
    for codes in df["evidence_codes"]:
        for c in (codes or []):
            code = c[0] if isinstance(c, list) else c
            counts[code] += 1
    return {code: counts.get(code, 0) / n for code in ALL_ACMG_CODES}


rule_freqs = rule_activation_frequencies(sandbox_results_df)
case_codes = sorted([c for c in ALL_ACMG_CODES if c in CASE_LEVEL_CODES], key=lambda c: -rule_freqs[c])
comp_codes = sorted([c for c in ALL_ACMG_CODES if c not in CASE_LEVEL_CODES], key=lambda c: -rule_freqs[c])
ordered_codes = case_codes + comp_codes
bar_colors = ["#e87ba4"] * len(case_codes) + ["#6da7ec"] * len(comp_codes)

fig, ax = plt.subplots(figsize=(7.5, 8))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

y_pos = np.arange(len(ordered_codes))
values = [rule_freqs[c] * 100 for c in ordered_codes]
bars = ax.barh(y_pos, values, color=bar_colors, height=0.65)
xmax = max(values) if max(values) > 0 else 1
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + xmax * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{v:.1f}%" if v > 0 else "0%", va="center", fontsize=8, color="#52514e")

ax.set_yticks(y_pos)
ax.set_yticklabels(ordered_codes)
ax.invert_yaxis()
ax.axhline(len(case_codes) - 0.5, color="#898781", linewidth=1)
ax.set_xlim(0, xmax * 1.12)

ax.set_xlabel("% of variants where code fired", color="#52514e")
ax.set_title(f"ACMG evidence code activation frequency (n={len(sandbox_results_df)})\n"
             "rose = case-level (patient-specific), blue = computational", fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
fig.tight_layout()
fig.savefig("outputs/sandbox_rule_activation.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()


## Tier 2: the 57 MYH7 "gold-standard" variants (ClinGen pilot / Whiffin et al. 2018)

Unlike the 100 conflicting variants above (no agreed truth to check against), this set has one: ClinGen's Cardiomyopathy pilot expert-panel curation of 57 MYH7 variants, and the *original* CardioClassifier's own call on the same 57 -- both recorded in `MYH7_57_variants_ClinGen_Cardioclassifier_comparison.csv`. The real per-variant ACMG evidence (which codes were curated, at what strength) lives in a second file, `MYH7_variants_case_rules_activation.csv` ("Supplementary Table 4").

The computational codes (PVS1, PM2, PP3/BP4, PM1, etc.) get computed the same way as everywhere else in this project -- live, from real VEP/gnomAD/ClinVar data. The **case-level** codes (PS2, PM6, PM3, BP2, PP1, BS4, PP4, BS2, BP5) can't be computed from any public API -- exactly the same limitation documented for the conflicting-variant set above -- so for this validation only, they're taken directly from the original curators' own findings (Supplementary Table 4) and fed in as `extra_answers`, the same override mechanism used for the manual-curation subset in `initial_validationn.ipynb`. This isolates the question: *given the same case-level facts a human curator had, does the pipeline's own computational evidence + scoring logic land on the same final call?*


### Parse and join the two source files

Both spreadsheets have a two-row header (a grouping row, then the real column names) -- `pandas` can't infer that automatically, so this reads them raw and reconstructs the header manually.


In [ ]:
# --------------------------------------------------
# Load MYH7_57_variants_ClinGen_Cardioclassifier_comparison.csv
# and MYH7_variants_case_rules_activation.csv ("Supplementary Table 4"),
# then join them on the shared cDNA notation (e.g. "c.428G>A").
# --------------------------------------------------
import pandas as pd

MYH7_DISEASE_CODE = "HCM-FAM"  # all 57 are from ClinGen's MYH7-HCM pilot curation
MYH7_TRANSCRIPT = "NM_000257.4"

# Source spreadsheets converted to CSV for this repo (single flat header row;
# the ClinGen/CardioClassifier 2-row group header was forward-filled at
# conversion, so "ClinGen activated rules_Case-level" etc. are already flat).
_comp = pd.read_csv("MYH7_57_variants_ClinGen_Cardioclassifier_comparison.csv")
_comp = _comp.rename(columns={"Variant": "cDNA"}).dropna(subset=["cDNA"])
_comp["cDNA"] = _comp["cDNA"].astype(str).str.strip()

# "Supplementary Table 4": header is the CSV's own first row; one Y/N column per
# ACMG code x strength tier (e.g. PP1 / PP1_Mod / "PP1_ Str").
_t4 = pd.read_csv("MYH7_variants_case_rules_activation.csv")
_t4["cDNA"] = _t4["cDNA"].astype(str).str.strip()
# The source spreadsheet is inconsistent about spacing in column names
# (e.g. "PP1_ Str" vs "PP1_Mod") -- normalize away whitespace/case so
# lookups below don't depend on getting that exactly right.
_T4_COLS_NORMALIZED = {str(c).replace(" ", "").upper(): c for c in _t4.columns}

myh7_variants_df = _comp.merge(_t4, on="cDNA", how="left", validate="one_to_one")
assert len(myh7_variants_df) == 57, f"expected 57 variants, got {len(myh7_variants_df)}"
assert myh7_variants_df["Sort"].notna().all(), "some variants didn't match between the two files"

myh7_variants_df["hgvs"] = f"{MYH7_TRANSCRIPT}(MYH7):" + myh7_variants_df["cDNA"]
print(f"{len(myh7_variants_df)}/57 variants matched between the two files")
myh7_variants_df[["cDNA", "hgvs", "Final classification_ClinGen", "Final classification_CardioClassifier"]].head()


### Translate each variant's case-level evidence into `extra_answers`

Each ACMG code in Supplementary Table 4 has one Y/N column *per strength tier* it can fire at (e.g. `PP1`, `PP1_Mod`, `PP1_ Str` for Supporting/Moderate/Strong segregation). This maps those columns onto the same `de_novo_answer` / `phasing_answer` / `segregation_answer` / `pp4_answer` / `bs2_answer` / `bp5_answer` keys `mainn.ipynb`'s curator-only cells already read -- passing a `segregation_meioses` count (7/5/3) chosen so the pipeline's *own* strength-grading logic (ClinGen Cardiomyopathy VCEP GN098 thresholds) reproduces the same strength the original curators used, rather than hardcoding the strength directly.


In [ ]:
# --------------------------------------------------
# Map each variant's Supplementary Table 4 row to extra_answers kwargs
# --------------------------------------------------
def _is_y(row, colname):
    val = row.get(_T4_COLS_NORMALIZED[colname.replace(" ", "").upper()])
    return str(val).strip().upper() == "Y"


def extra_answers_from_table4_row(row):
    answers = {}

    # De novo status: PS2 (confirmed) / PM6 (assumed) are mutually exclusive.
    if _is_y(row, "PS2"):
        answers["de_novo_answer"], answers["parentage_answer"] = "y", "y"
    elif _is_y(row, "PM6"):
        answers["de_novo_answer"], answers["parentage_answer"] = "y", "n"

    # Phasing: PM3 (trans with a pathogenic variant) / BP2 (cis) are mutually exclusive.
    if _is_y(row, "PM3"):
        answers["phasing_answer"], answers["phasing_relationship"] = "y", "trans"
    elif _is_y(row, "BP2"):
        answers["phasing_answer"], answers["phasing_relationship"] = "y", "cis"

    # Segregation: PP1 (3 strength tiers) / BS4 (non-segregation) are mutually exclusive.
    if _is_y(row, "PP1_ Str"):
        answers.update(segregation_answer="y", segregation_result="segregates", segregation_meioses=7)
    elif _is_y(row, "PP1_Mod"):
        answers.update(segregation_answer="y", segregation_result="segregates", segregation_meioses=5)
    elif _is_y(row, "PP1"):
        answers.update(segregation_answer="y", segregation_result="segregates", segregation_meioses=3)
    elif _is_y(row, "BS4"):
        answers.update(segregation_answer="y", segregation_result="does_not_segregate", segregation_meioses=2)

    # Independent of the above: phenotype specificity, healthy-adult observation, alternate cause.
    if _is_y(row, "PP4"):
        answers["pp4_answer"] = "y"
    if _is_y(row, "BS2"):
        answers["bs2_answer"] = "y"
    if _is_y(row, "BP5"):
        answers["bp5_answer"] = "y"

    return answers


myh7_variants_df["extra_answers"] = myh7_variants_df.apply(extra_answers_from_table4_row, axis=1)

n_with_case_evidence = (myh7_variants_df["extra_answers"].map(len) > 0).sum()
print(f"{n_with_case_evidence}/{len(myh7_variants_df)} variants have at least one case-level code injected")
myh7_variants_df[["cDNA", "extra_answers"]].head(10)


### Run all 57 through the real pipeline

Same resumable-checkpoint pattern as the conflicting-variant batch above, its own checkpoint file. `disease_short_code` is fixed to `"HCM-FAM"` for every variant (this whole set is ClinGen's MYH7-HCM pilot), so `infer_disease_short_code` isn't needed here.


In [ ]:
# --------------------------------------------------
# Resumable batch runner for the 57 MYH7 variants
# --------------------------------------------------
import time
from pathlib import Path

MYH7_CHECKPOINT_PATH = Path("outputs/myh7_57_batch.jsonl")
MYH7_PAUSE_BETWEEN_VARIANTS = 2.0  # each variant now makes up to ~2 gnomAD calls (combined FAF95+PS4 query, occasionally a coverage check) -- a longer pause reduces sustained request rate across the whole 57-variant batch, since gnomAD rate-limited us under the old 0.5s pacing even with per-call retry/backoff
MYH7_CHECKPOINT_PATH.parent.mkdir(exist_ok=True)

_succeeded_before = set()
if MYH7_CHECKPOINT_PATH.exists():
    with MYH7_CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "error" not in rec:
                _succeeded_before.add(rec["hgvs"])
    print(f"Resuming: {len(_succeeded_before)} variants already succeeded in {MYH7_CHECKPOINT_PATH}")

n_ok, n_failed = 0, 0
with MYH7_CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt:
    for i, row in myh7_variants_df.reset_index(drop=True).iterrows():
        hgvs = row["hgvs"]
        if hgvs in _succeeded_before:
            continue

        try:
            result_row = run_variant_through_main1(hgvs, MYH7_DISEASE_CODE, **row["extra_answers"])
        except Exception as exc:
            n_failed += 1
            ckpt.write(json.dumps({"hgvs": hgvs, "cDNA": row["cDNA"], "error": str(exc)}) + "\n")
            ckpt.flush()
            print(f"[{i+1}/{len(myh7_variants_df)}] FAILED {hgvs}: {exc}")
            time.sleep(MYH7_PAUSE_BETWEEN_VARIANTS)
            continue

        result_row.update({
            "cDNA": row["cDNA"],
            "clingen_final": row["Final classification_ClinGen"],
            "original_cardioclassifier_final": row["Final classification_CardioClassifier"],
        })
        ckpt.write(json.dumps(result_row, default=str) + "\n")
        ckpt.flush()
        n_ok += 1
        print(f"[{i+1}/{len(myh7_variants_df)}] {hgvs} -> {result_row['pipeline_classification']} "
              f"(ClinGen: {row['Final classification_ClinGen']})")
        time.sleep(MYH7_PAUSE_BETWEEN_VARIANTS)

print(f"\nDone. {n_ok} succeeded, {n_failed} failed this run. Full results in {MYH7_CHECKPOINT_PATH}")


### Does our pipeline agree with ClinGen and the original CardioClassifier?


In [ ]:
# --------------------------------------------------
# Load the checkpoint and compare against both ground-truth columns
# --------------------------------------------------
_records = []
with MYH7_CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            _records.append(json.loads(line))

myh7_results_df = pd.DataFrame(_records)
myh7_results_df = myh7_results_df[myh7_results_df.get("error").isna()] if "error" in myh7_results_df.columns else myh7_results_df
print(f"{len(myh7_results_df)}/57 variants classified successfully")

myh7_results_df["matches_clingen"] = myh7_results_df["pipeline_classification"] == myh7_results_df["clingen_final"]
myh7_results_df["matches_original_cc"] = myh7_results_df["pipeline_classification"] == myh7_results_df["original_cardioclassifier_final"]

n = len(myh7_results_df)
print(f"Agrees with ClinGen expert panel:        {myh7_results_df['matches_clingen'].sum()}/{n} "
      f"({myh7_results_df['matches_clingen'].mean():.0%})")
print(f"Agrees with original CardioClassifier:    {myh7_results_df['matches_original_cc'].sum()}/{n} "
      f"({myh7_results_df['matches_original_cc'].mean():.0%})")

print("\nDiscordant with ClinGen:")
myh7_results_df[~myh7_results_df["matches_clingen"]][
    ["cDNA", "pipeline_classification", "clingen_final", "original_cardioclassifier_final", "evidence_codes"]
]


In [ ]:
# --------------------------------------------------
# Grouped bar chart: pipeline vs ClinGen vs original CardioClassifier
# --------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

_TIER_ORDER = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]
_SERIES = [
    ("ClinGen expert panel", "clingen_final", "#e1af64"),
    ("Original CardioClassifier", "original_cardioclassifier_final", "#86b6ef"),
    ("This pipeline", "pipeline_classification", "#104281"),
]

fig, ax = plt.subplots(figsize=(8, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

_x = np.arange(len(_TIER_ORDER))
_width = 0.26
for i, (label, col, color) in enumerate(_SERIES):
    counts = myh7_results_df[col].value_counts().reindex(_TIER_ORDER, fill_value=0)
    ax.bar(_x + (i - 1) * _width, counts.values, width=_width, color=color, label=label)

ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_title(f"MYH7 gold-standard set (n={len(myh7_results_df)}): three classifiers, same variants",
             fontsize=11, color="#0b0b0b")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig("outputs/myh7_57_three_way_comparison.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()


### Rule-by-rule audit: which codes are we missing or adding, per variant?

The chart above says *how many* variants land in each tier -- it can't say *why*. This builds the actual per-variant, per-code diff: for every one of the 57, which ACMG codes did ClinGen's own curation activate that we don't fire (`missing`), and which do we fire that ClinGen's summary doesn't list (`extra`). Saved to CSV so each row -- and the real evidence behind it (`evidence_codes`, `outputs/<variant>_classification.json`) -- can be checked by hand rather than trusted from an aggregate count.


In [ ]:
# --------------------------------------------------
# Per-variant, per-code diff vs ClinGen's own activated-rules summary
# --------------------------------------------------
import re

def _split_codes(s):
    """'PS4, PM1, PM2, PP3' -> {'PS4','PM1','PM2','PP3'}; strips _strong/_moderate/etc suffixes."""
    if not isinstance(s, str) or s.strip() in ("-", ""):
        return set()
    codes = set()
    for c in s.split(","):
        c = c.strip()
        if not c or c == "-":
            continue
        codes.add(re.split(r"_", c)[0])
    return codes


_audit_df = myh7_variants_df.merge(
    pd.DataFrame(
        [json.loads(l) for l in MYH7_CHECKPOINT_PATH.open("r", encoding="utf-8") if l.strip()]
    ).pipe(lambda d: d[d.get("error").isna()] if "error" in d.columns else d).drop(columns=["cDNA"], errors="ignore"),
    on="hgvs", how="left",
)
_audit_df = _audit_df[_audit_df["pipeline_classification"].notna()].copy()

rows = []
for _, row in _audit_df.iterrows():
    clingen_codes = (_split_codes(row["ClinGen activated rules_Computational"])
                      | _split_codes(row["ClinGen activated rules_Case-level"]))
    our_codes = set(c[0] if isinstance(c, list) else c for c in (row["evidence_codes"] or []))
    rows.append({
        "cDNA": row["cDNA"],
        "clingen_final": row["Final classification_ClinGen"],
        "original_cc_final": row["Final classification_CardioClassifier"],
        "pipeline_final": row["pipeline_classification"],
        "concordant": row["pipeline_classification"] == row["Final classification_ClinGen"],
        "clingen_codes": ", ".join(sorted(clingen_codes)) or "-",
        "our_codes": ", ".join(sorted(our_codes)) or "-",
        "missing": ", ".join(sorted(clingen_codes - our_codes)) or "-",
        "extra": ", ".join(sorted(our_codes - clingen_codes)) or "-",
    })

rule_audit_df = pd.DataFrame(rows)
rule_audit_df.to_csv("outputs/myh7_rule_audit.csv", index=False)
print(f"{len(rule_audit_df)} variants audited -> outputs/myh7_rule_audit.csv")
print(f"{(rule_audit_df['missing'] != '-').sum()} have >=1 missing code, "
      f"{(rule_audit_df['extra'] != '-').sum()} have >=1 extra code")

rule_audit_df


### Which specific codes are driving the discordances?

Same data, aggregated by code instead of by variant -- how many of the 55 variants is each code missing from / extra in.


In [ ]:
# --------------------------------------------------
# Aggregate missing/extra frequency by ACMG code
# --------------------------------------------------
from collections import Counter

missing_counter, extra_counter = Counter(), Counter()
for _, row in rule_audit_df.iterrows():
    if row["missing"] != "-":
        missing_counter.update(row["missing"].split(", "))
    if row["extra"] != "-":
        extra_counter.update(row["extra"].split(", "))

_all_codes = sorted(set(missing_counter) | set(extra_counter), key=lambda c: -(missing_counter[c] + extra_counter[c]))
freq_df = pd.DataFrame({
    "code": _all_codes,
    "missing_in_n_variants": [missing_counter[c] for c in _all_codes],
    "extra_in_n_variants": [extra_counter[c] for c in _all_codes],
})

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
_y = np.arange(len(freq_df))
_h = 0.35
ax.barh(_y + _h/2, freq_df["missing_in_n_variants"], height=_h, color="#e87ba4", label="ClinGen fired, we didn't (missing)")
ax.barh(_y - _h/2, freq_df["extra_in_n_variants"], height=_h, color="#6da7ec", label="We fired, ClinGen's summary doesn't list (extra)")
ax.set_yticks(_y)
ax.set_yticklabels(freq_df["code"])
ax.invert_yaxis()
ax.set_xlabel(f"Number of variants (of {len(rule_audit_df)})", color="#52514e")
ax.set_title("Per-code discordance frequency, MYH7 gold-standard set", fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=9, loc="lower right")
fig.tight_layout()
fig.savefig("outputs/myh7_rule_audit_frequency.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

freq_df


### Is the 2018 gold-standard comparison even fair? Checking ClinGen's 2026 classification

The benchmark above compares the CardioClassifier reimplementation (live gnomAD, ClinVar, and REVEL) against a frozen 2018 snapshot of ClinGen's curation. That is not the contemporaneous comparison Whiffin et al. 2018 themselves made (their 2018 tool vs. 2018 ClinGen). Some of the apparent discordance may reflect elapsed time rather than reimplementation error.

**Methodological note:** ClinVar's aggregate (`esummary`) classification is not the right comparator here -- it reflects every submitter combined (often "Conflicting classifications" purely because non-expert labs disagree), which is a different, noisier signal than the 2018 comparison (specifically the ClinGen Cardiomyopathy Expert Panel's own assertion). Confirmed this matters with a real example: `c.3286G>T`'s aggregate is diluted by eight disagreeing submitters, while the ClinGen Cardiomyopathy Expert Panel itself (OrgID 506161 -- the same panel behind the original 57-variant gold standard) currently classifies it as Likely Benign. This pulls that specific panel's 2026 assertion via `efetch rettype=vcv`, parsing `<ClinicalAssertion>` blocks filtered to `SubmitterName="ClinGen Cardiomyopathy Variant Curation Expert Panel"` -- the same technique used above for the conflicting-variant submitter breakdown.


In [ ]:
# --------------------------------------------------
# Concordance vs. the 2018 snapshot vs. the SAME expert panel's current assertion
# --------------------------------------------------
# Self-contained (defines its own NCBI helpers) rather than reusing
# _esearch/_ncbi_get from the conflicting-variant section far above --
# confirmed real failure mode: after a kernel restart, re-running just the
# MYH7 cells without re-running everything above them left those
# undefined. EUTILS/re/time/pd are standard-library-adjacent and already
# guaranteed available by this point regardless of run order.
import re
import time
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
EXPERT_PANEL_NAME = "ClinGen Cardiomyopathy Variant Curation Expert Panel"


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    """Same retry/backoff pattern used throughout this project for NCBI eutils calls."""
    resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt == max_retries:
                raise
            time.sleep(backoff_base ** attempt)
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            time.sleep(backoff_base ** attempt)
            continue
        resp.raise_for_status()
        return resp
    return resp


def _esearch(term, retmax):
    resp = _ncbi_get(f"{EUTILS}/esearch.fcgi", params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return resp.json().get("esearchresult", {}).get("idlist", [])


def _parse_expert_panel_assertion(vcv_xml):
    for block in re.findall(r"<ClinicalAssertion .*?</ClinicalAssertion>", vcv_xml, re.S):
        submitter_m = re.search(r'SubmitterName="([^"]*)"', block)
        if not submitter_m or submitter_m.group(1) != EXPERT_PANEL_NAME:
            continue
        classification_m = re.search(r"<GermlineClassification>([^<]*)</GermlineClassification>", block)
        date_m = re.search(r'DateLastEvaluated="([^"]*)"', block)
        return {
            "classification": classification_m.group(1) if classification_m else None,
            "last_evaluated": date_m.group(1) if date_m else None,
        }
    return None


def fetch_expert_panel_current(gene, cdna):
    ids = _esearch(f"{gene}[gene] AND {cdna}", retmax=1)
    if not ids:
        return None
    r = _ncbi_get(f"{EUTILS}/efetch.fcgi",
                  params={"db": "clinvar", "id": ids[0], "rettype": "vcv",
                          "is_variationid": "true", "retmode": "xml"})
    return _parse_expert_panel_assertion(r.text)


_rows = []
for _, row in rule_audit_df.iterrows():
    try:
        _rows.append(fetch_expert_panel_current("MYH7", row["cDNA"]))
    except Exception as exc:
        print(f"  failed for {row['cDNA']}: {exc}")
        _rows.append(None)
    time.sleep(0.4)

rule_audit_df["expert_panel_current_significance"]   = [r["classification"] if r else None for r in _rows]
rule_audit_df["expert_panel_current_last_evaluated"]  = [r["last_evaluated"] if r else None for r in _rows]
rule_audit_df.to_csv("outputs/myh7_expert_panel_current.csv", index=False)


def _normalize_sig(s):
    if not isinstance(s, str):
        return None
    s = s.strip().lower()
    return "vus" if "uncertain" in s else s.replace(" ", "_")


rule_audit_df["current_norm"]     = rule_audit_df["expert_panel_current_significance"].map(_normalize_sig)
rule_audit_df["pipeline_norm"]    = rule_audit_df["pipeline_final"].str.strip().str.lower().str.replace(" ", "_")
rule_audit_df["clingen2018_norm"] = rule_audit_df["clingen_final"].str.strip().str.lower().str.replace(" ", "_")

_valid = rule_audit_df.dropna(subset=["current_norm"]).copy()
print(f"{len(_valid)}/{len(rule_audit_df)} variants: found a 2026 ClinGen Cardiomyopathy Expert Panel assertion\n")

_conc_2018    = (_valid["pipeline_norm"] == _valid["clingen2018_norm"]).mean()
_conc_current = (_valid["pipeline_norm"] == _valid["current_norm"]).mean()
print(f"Concordance vs 2018 ClinGen snapshot   (same {len(_valid)}): "
      f"{(_valid['pipeline_norm']==_valid['clingen2018_norm']).sum()}/{len(_valid)} ({_conc_2018:.1%})")
print(f"Concordance vs 2026 expert panel assertion: "
      f"{(_valid['pipeline_norm']==_valid['current_norm']).sum()}/{len(_valid)} ({_conc_current:.1%})")

_reclassified = _valid[_valid["clingen2018_norm"] != _valid["current_norm"]]
print(f"\n{len(_reclassified)}/{len(_valid)} variants: the expert panel ITSELF changed its own call since 2018 "
      f"-- ACMG classification is a living process, not a fixed ground truth.")

_reclassified[["cDNA", "clingen_final", "expert_panel_current_significance", "pipeline_final",
               "expert_panel_current_last_evaluated"]]


### PS4: a genuine, expected limitation -- not something further code can fix

Unlike BS1, the remaining PS4 gap doesn't resolve against ClinGen's current record -- it's a real case-data completeness limit, and the original paper hits the exact same wall:

> "differences in PS4 and PM5 are due to the increased availability of proband data to the ClinGen team (not available from public repositories)" -- Whiffin et al. 2018

Even the original authors, with direct access to unpublished proband data, could not close this gap using public sources alone (their own benchmark: 7/157 computational-rule discordances were PS4). Our remaining 10 break down the same way:
- **3 variants** (`c.2608C>T`, `c.3133C>T`, `c.4258C>T`) have real, combined case+control data (JUL_HCMgenes.tsv + the original cardiodb.org LMM/RBH/ORGL cohorts) -- the odds ratio's lower 95% CI (3.7-4.5) is genuinely just short of the Supporting threshold (5). Not a bug: correctly computed, just not quite enough public evidence.
- **7 variants** have no case data in either public source at all.

Documented here as a limitation, not chased further in code: the private proband data ClinGen used for these specific variants simply has no public equivalent to substitute.


## Two contemporaneous comparisons, not one mixed one

Comparing a live pipeline against a frozen 2018 snapshot conflates two different questions into one number. Splitting it into two internally-consistent comparisons instead:

1. **ClinGen (2018) vs. the original CardioClassifier** -- both fixed at 2018, matching Whiffin et al.'s own reported benchmark. No live lookups required; already present in the source spreadsheet.
2. **ClinGen (2026) vs. the CardioClassifier reimplementation** -- both current, both live. This is the fair test of whether the reimplementation, evaluated today, agrees with today's expert consensus.

Neither comparison is "the" validation number on its own -- together they separate "did the reimplementation faithfully reproduce the 2018 benchmark" from "does the reimplementation track present-day expert opinion."


In [ ]:
# --------------------------------------------------
# Comparison 1: 2018 ClinGen vs. the original CardioClassifier (both 2018)
# --------------------------------------------------
_TIER_ORDER = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]

_counts_clingen_2018 = myh7_variants_df["Final classification_ClinGen"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_counts_origcc_2018  = myh7_variants_df["Final classification_CardioClassifier"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_concordance_2018 = (
    myh7_variants_df["Final classification_ClinGen"] == myh7_variants_df["Final classification_CardioClassifier"]
).mean()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

_x = np.arange(len(_TIER_ORDER))
_width = 0.35
ax.bar(_x - _width/2, _counts_clingen_2018.values, width=_width, color="#e1af64", label="ClinGen (2018)")
ax.bar(_x + _width/2, _counts_origcc_2018.values, width=_width, color="#86b6ef", label="Original CardioClassifier (2018)")

ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_title(f"2018: ClinGen vs. original CardioClassifier (n=57, {_concordance_2018:.0%} concordant)",
             fontsize=11, color="#0b0b0b")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig("outputs/myh7_comparison_2018.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(f"Concordance: {int((myh7_variants_df['Final classification_ClinGen']==myh7_variants_df['Final classification_CardioClassifier']).sum())}/57 "
      f"({_concordance_2018:.1%}) -- matches Whiffin et al. 2018's own reported 50/57 (87.7%).")


### Comparison 2: ClinGen (2026) vs. the CardioClassifier reimplementation

Uses `outputs/myh7_expert_panel_current.csv` (the ClinGen Cardiomyopathy Expert Panel's own 2026 assertion, pulled live) against the CardioClassifier reimplementation's own live classification. Restricted to the **46/57** variants where a 2026 expert-panel assertion was found -- labelled explicitly rather than silently padding the denominator.


In [ ]:
# --------------------------------------------------
# Comparison 2: CURRENT ClinGen expert panel vs. THIS reimplementation (both current)
# --------------------------------------------------
_current_df = pd.read_csv("outputs/myh7_expert_panel_current.csv")

_label_map = {"pathogenic": "Pathogenic", "likely pathogenic": "Likely Pathogenic",
              "uncertain significance": "VUS", "likely benign": "Likely Benign", "benign": "Benign"}


def _normalize_title(s):
    if not isinstance(s, str):
        return None
    key = s.strip().lower()
    return "VUS" if "uncertain" in key else _label_map.get(key, s.strip())


_current_df["current_clingen_tier"] = _current_df["expert_panel_current_significance"].map(_normalize_title)
_current_valid = _current_df.dropna(subset=["current_clingen_tier"]).copy()

_counts_clingen_now = _current_valid["current_clingen_tier"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_counts_pipeline_now = _current_valid["pipeline_final"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_concordance_now = (_current_valid["current_clingen_tier"] == _current_valid["pipeline_final"]).mean()

fig, ax = plt.subplots(figsize=(8.5, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

_x = np.arange(len(_TIER_ORDER))
_width = 0.35
ax.bar(_x - _width/2, _counts_clingen_now.values, width=_width, color="#e1af64", label="ClinGen (2026)")
ax.bar(_x + _width/2, _counts_pipeline_now.values, width=_width, color="#104281", label="CardioClassifier reimplementation (2026)")

ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_title(f"ClinGen (2026) vs. CardioClassifier reimplementation ({_concordance_now:.0%} concordant)\n"
             f"same {len(_current_valid)} variants both sides -- {57-len(_current_valid)}/57 excluded (no 2026 expert-panel record found)",
             fontsize=10, color="#0b0b0b")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig("outputs/myh7_comparison_current.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

_n_missing = len(_current_df) - len(_current_valid)
print(f"Concordance: {int((_current_valid['current_clingen_tier']==_current_valid['pipeline_final']).sum())}/{len(_current_valid)} "
      f"({_concordance_now:.1%})")
print(f"({_n_missing}/57 variants excluded -- no 2026 ClinGen Cardiomyopathy Expert Panel assertion found for them)")


### Comparison 3: all 57 variants, ClinGen's best available position

Comparison 2 restricted to the 46/57 variants with a findable 2026 expert-panel record, excluding the other 11 entirely. But "no 2026 record found" most likely means ClinGen simply has not revisited that variant since 2018 -- not that the comparison is invalid. This extends comparison 2 to all 57: **ClinGen's 2026 assertion where one was found, falling back to the 2018 classification for the 11 where it wasn't** (on the assumption that an unrevised variant's 2018 call is still its best-available current position).

**Caveat, stated plainly:** "no 2026 record found" is not proof ClinGen hasn't revisited the variant -- it could occasionally reflect a gap in this retrieval rather than a genuine absence of a newer record. This is the most likely explanation, not a guaranteed one.


In [ ]:
# --------------------------------------------------
# Comparison 3: ClinGen 2026-where-found, else 2018 fallback vs. the reimplementation (n=57)
# --------------------------------------------------
_fallback_df = pd.read_csv("outputs/myh7_expert_panel_current.csv")


def _normalize_tier(s):
    if not isinstance(s, str):
        return None
    key = s.strip().lower()
    return "VUS" if "uncertain" in key else _label_map.get(key, s.strip())


_fallback_df["current_tier"] = _fallback_df["expert_panel_current_significance"].map(_normalize_tier)
_fallback_df["clingen_2018_tier"] = _fallback_df["clingen_final"]
_fallback_df["clingen_best_available"] = _fallback_df["current_tier"].fillna(_fallback_df["clingen_2018_tier"])

_n_fallback = _fallback_df["current_tier"].isna().sum()
_concordance_fallback = (_fallback_df["pipeline_final"] == _fallback_df["clingen_best_available"]).mean()

_counts_clingen_fb = _fallback_df["clingen_best_available"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_counts_pipeline_fb = _fallback_df["pipeline_final"].value_counts().reindex(_TIER_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

_x = np.arange(len(_TIER_ORDER))
_width = 0.35
ax.bar(_x - _width/2, _counts_clingen_fb.values, width=_width, color="#e1af64",
       label="ClinGen (2026, else 2018 fallback)")
ax.bar(_x + _width/2, _counts_pipeline_fb.values, width=_width, color="#104281",
       label="CardioClassifier reimplementation (2026)")

ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_title(f"ClinGen best-available position vs. CardioClassifier reimplementation ({_concordance_fallback:.0%} concordant)\n"
             f"all 57 variants -- {_n_fallback}/57 used the 2018 fallback (no 2026 record found)",
             fontsize=10, color="#0b0b0b")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig("outputs/myh7_comparison_fallback.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(f"Concordance: {int((_fallback_df['pipeline_final']==_fallback_df['clingen_best_available']).sum())}/57 "
      f"({_concordance_fallback:.1%})")
print(f"({_n_fallback}/57 used the 2018 fallback; of those, "
      f"{int((_fallback_df[_fallback_df['current_tier'].isna()]['pipeline_final']==_fallback_df[_fallback_df['current_tier'].isna()]['clingen_2018_tier']).sum())}/{_n_fallback} concordant)")


### Why are these 15 discordant? Root-cause breakdown

Each of the 15 variants above was individually investigated (evidence codes, scores, ClinVar re-evaluation dates -- not inferred from aggregate counts). Colour encodes which direction the mismatch runs: **rose = this pipeline calls the variant less pathogenic than ClinGen's current position**, **blue = this pipeline calls it more pathogenic**.


In [ ]:
# --------------------------------------------------
# Root-cause breakdown of the 15 discordant variants (individually verified above)
# --------------------------------------------------
_ps4_gap = {"c.2207T>C", "c.2513C>T", "c.2608C>T", "c.4258C>T", "c.5401G>A"}
_post2018_evidence = {"c.2539A>G", "c.4066G>A", "c.4130C>T", "c.3169G>A", "c.3578G>A", "c.5135G>A"}
_opp_boundary = {"c.4276G>A"}
_combined_case_data_generous = {"c.1157A>G"}
_bs1_related = {"c.5326A>G", "c.5329G>A"}

_CATEGORY_INFO = {
    "Evidence ClinGen incorporated\nafter 2018":                          (_post2018_evidence, "under"),
    "PS4: private case-cohort data\nunavailable to the reimplementation": (_ps4_gap, "under"),
    "BS1: outdated review or\nborderline population frequency":          (_bs1_related, "under"),
    "Reimplementation case data exceeds\nClinGen's cited evidence":       (_combined_case_data_generous, "over"),
    "Posterior-threshold correction\n(residual boundary case)":          (_opp_boundary, "over"),
}

_cat_counts = {label: len(cdnas) for label, (cdnas, _) in _CATEGORY_INFO.items()}
_cat_direction = {label: direction for label, (_, direction) in _CATEGORY_INFO.items()}
_ordered_labels = sorted(_cat_counts, key=lambda l: -_cat_counts[l])

_DIRECTION_COLORS = {"under": "#e87ba4", "over": "#6da7ec"}
_colors = [_DIRECTION_COLORS[_cat_direction[l]] for l in _ordered_labels]
_values = [_cat_counts[l] for l in _ordered_labels]

fig, ax = plt.subplots(figsize=(9.5, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

_y = range(len(_ordered_labels))
bars = ax.barh(_y, _values, color=_colors, height=0.6)
for bar, v in zip(bars, _values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2, str(v),
            va="center", fontsize=10, color="#0b0b0b")

ax.set_yticks(_y)
ax.set_yticklabels(_ordered_labels, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlim(0, max(_values) * 1.2)
ax.set_xlabel("Number of variants (of 15 discordant)", color="#52514e")
ax.set_title("Root cause of every discordant call\nClinGen (2026) vs. CardioClassifier reimplementation (n=46, 15 discordant)",
             fontsize=10.5, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")

_legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=_DIRECTION_COLORS["under"], label="Reimplementation less pathogenic than ClinGen"),
    plt.Rectangle((0, 0), 1, 1, color=_DIRECTION_COLORS["over"], label="Reimplementation more pathogenic than ClinGen"),
]
ax.legend(handles=_legend_handles, loc="lower right", frameon=False, fontsize=8.5)

fig.tight_layout()
fig.savefig("outputs/myh7_discordance_root_causes.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(f"{sum(_values)}/15 discordant variants categorized")
print(f"13/15 (87%) trace to ClinGen having an information advantage the CardioClassifier reimplementation "
      f"structurally cannot replicate (private proband data + post-2018 literature); 2/15 are the reimplementation "
      f"being MORE confident than ClinGen's 2026 position, one of which is a direct, disclosed side effect of "
      f"the O_PP precision correction.")


### What evidence actually justified the post-2018 reclassifications?

The "evidence ClinGen incorporated after 2018" category (6 variants) is a claim, not just a label, and is worth substantiating with the expert panel's own stated reasoning. The VCV record already retrieved for the 2026-classification comparison includes each submitter's full `<Comment>` text and cited PMIDs; this extracts that text specifically for the ClinGen Cardiomyopathy Variant Curation Expert Panel's own assertion (not any other submitter) for these six variants.

(Parsing note: one record uses `<Comment Type="public">`, another uses a bare `<Comment>` with no attribute. The regex matches both.)


In [ ]:
# --------------------------------------------------
# Full evidence text behind the "post-2018 evidence" reclassifications
# --------------------------------------------------
# Self-contained (see myh7_current_expert_panel_01's docstring for why --
# same real failure mode: re-running just this cell after a kernel restart
# without re-running everything above it).
import re
import time
import requests

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
EXPERT_PANEL_NAME = "ClinGen Cardiomyopathy Variant Curation Expert Panel"


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt == max_retries:
                raise
            time.sleep(backoff_base ** attempt)
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            time.sleep(backoff_base ** attempt)
            continue
        resp.raise_for_status()
        return resp
    return resp


def _esearch(term, retmax):
    resp = _ncbi_get(f"{EUTILS}/esearch.fcgi", params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return resp.json().get("esearchresult", {}).get("idlist", [])


def fetch_expert_panel_comment(gene, cdna):
    ids = _esearch(f"{gene}[gene] AND {cdna}", retmax=1)
    if not ids:
        return None
    r = _ncbi_get(f"{EUTILS}/efetch.fcgi",
                  params={"db": "clinvar", "id": ids[0], "rettype": "vcv",
                          "is_variationid": "true", "retmode": "xml"})
    for block in re.findall(r"<ClinicalAssertion .*?</ClinicalAssertion>", r.text, re.S):
        if f'SubmitterName="{EXPERT_PANEL_NAME}"' not in block:
            continue
        comment_m = re.search(r"<Comment[^>]*>(.*?)</Comment>", block, re.S)  # Type attr is optional
        classification_m = re.search(r"<GermlineClassification>([^<]*)</GermlineClassification>", block)
        date_m = re.search(r'DateLastEvaluated="([^"]*)"', block)
        return {
            "classification": classification_m.group(1) if classification_m else None,
            "date": date_m.group(1) if date_m else None,
            "comment": comment_m.group(1) if comment_m else None,
        }
    return None


_post2018_variants = ["c.2539A>G", "c.4066G>A", "c.4130C>T", "c.3169G>A", "c.3578G>A", "c.5135G>A"]
expert_panel_evidence = {}
for _v in _post2018_variants:
    expert_panel_evidence[_v] = fetch_expert_panel_comment("MYH7", _v)
    time.sleep(0.4)

_n_pers_comm = 0
for _v, _info in expert_panel_evidence.items():
    print(f"=== {_v} ({_info['classification']}, evaluated {_info['date']}) ===")
    print(_info["comment"])
    _n_pers_comm += (_info["comment"] or "").count("pers. comm")
    print()

print(f"Across all 6: {_n_pers_comm} separate 'pers. comm.' (unpublished, private lab communication) "
      f"citations -- evidence that structurally cannot be found via any public literature search, "
      f"not just evidence this pipeline happened to miss.")


### Rules activated by ClinGen (2026) vs. rules activated by the reimplementation

The comparisons above are all at the level of *final classification* (Pathogenic/VUS/etc.). This goes one level deeper: for the 46 variants with a findable 2026 record, ClinGen's own free-text comment states exactly which ACMG/AMP codes they applied (e.g. "...criteria applied: PS4, PP1_Strong, PM2, PP3"). This extracts that code list -- via the same VCV comment already pulled for the evidence-text section above -- and compares it directly against the reimplementation's own `evidence_codes`, the same way the 2018 comparison was done at the top of this notebook, but now against ClinGen's *current* stated reasoning rather than the 2018 spreadsheet.


In [ ]:
# --------------------------------------------------
# Rule-level comparison: ClinGen's 2026 stated criteria vs. the reimplementation's own codes
# --------------------------------------------------
# Self-contained: loads its own copies of everything needed rather than
# depending on variable names/state from earlier cells (see
# myh7_current_expert_panel_01's docstring for why -- confirmed real
# failure mode after a kernel restart).
import re
import time
import pickle
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from collections import Counter

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
EXPERT_PANEL_NAME = "ClinGen Cardiomyopathy Variant Curation Expert Panel"
_CODE_RE = re.compile(r'\b((?:PVS|PS|PM|PP|BA|BS|BP)\d)(?:_\w+)?\b')


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    resp = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, timeout=30)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt == max_retries:
                raise
            time.sleep(backoff_base ** attempt)
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            time.sleep(backoff_base ** attempt)
            continue
        resp.raise_for_status()
        return resp
    return resp


def _esearch(term, retmax):
    resp = _ncbi_get(f"{EUTILS}/esearch.fcgi", params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return resp.json().get("esearchresult", {}).get("idlist", [])


rule_audit_df = pd.read_csv("outputs/myh7_rule_audit.csv")
current_df = pd.read_csv("outputs/myh7_expert_panel_current.csv")


def fetch_expert_panel_codes(gene, cdna):
    ids = _esearch(f"{gene}[gene] AND {cdna}", retmax=1)
    if not ids:
        return None
    r = _ncbi_get(f"{EUTILS}/efetch.fcgi",
                  params={"db": "clinvar", "id": ids[0], "rettype": "vcv",
                          "is_variationid": "true", "retmode": "xml"})
    for block in re.findall(r"<ClinicalAssertion .*?</ClinicalAssertion>", r.text, re.S):
        if f'SubmitterName="{EXPERT_PANEL_NAME}"' not in block:
            continue
        comment_m = re.search(r"<Comment[^>]*>(.*?)</Comment>", block, re.S)
        if not comment_m:
            return set()
        comment = comment_m.group(1)
        summary_m = re.search(r"In summary.*", comment, re.S)
        summary = summary_m.group(0) if summary_m else comment
        return set(_CODE_RE.findall(summary))
    return None


_valid_cdnas = current_df[current_df["expert_panel_current_significance"].notna()]["cDNA"].tolist()
expert_panel_codes = {}
for _i, _cdna in enumerate(_valid_cdnas):
    try:
        expert_panel_codes[_cdna] = fetch_expert_panel_codes("MYH7", _cdna)
    except Exception as _exc:
        print(f"  failed for {_cdna}: {_exc}")
        expert_panel_codes[_cdna] = None
    time.sleep(0.4)

with open("outputs/myh7_expert_panel_codes.pkl", "wb") as f:
    pickle.dump(expert_panel_codes, f)

rule_audit_df["our_codes_set"] = rule_audit_df["our_codes"].apply(
    lambda s: set(c.strip() for c in s.split(",")) if isinstance(s, str) and s.strip() != "-" else set()
)

_missing_counter, _extra_counter = Counter(), Counter()
_n_compared = 0
for _cdna, _clingen_codes in expert_panel_codes.items():
    if _clingen_codes is None:
        continue
    _row = rule_audit_df[rule_audit_df["cDNA"] == _cdna]
    if _row.empty:
        continue
    _our_codes = _row.iloc[0]["our_codes_set"]
    _missing_counter.update(_clingen_codes - _our_codes)
    _extra_counter.update(_our_codes - _clingen_codes)
    _n_compared += 1

print(f"Compared rule-by-rule across {_n_compared} variants (of {len(_valid_cdnas)} with a findable 2026 record)\n")

_all_codes = sorted(set(_missing_counter) | set(_extra_counter),
                     key=lambda c: -(_missing_counter[c] + _extra_counter[c]))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
_y = np.arange(len(_all_codes))
_h = 0.35
ax.barh(_y + _h/2, [_missing_counter[c] for c in _all_codes], height=_h, color="#e87ba4",
        label="ClinGen (2026) applied, reimplementation didn't")
ax.barh(_y - _h/2, [_extra_counter[c] for c in _all_codes], height=_h, color="#6da7ec",
        label="Reimplementation applied, ClinGen (2026) doesn't state")
ax.set_yticks(_y)
ax.set_yticklabels(_all_codes)
ax.invert_yaxis()
ax.set_xlabel(f"Number of variants (of {_n_compared} compared)", color="#52514e")
ax.set_title("Rule-level discordance: ClinGen (2026) stated criteria vs. reimplementation",
             fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=8.5, loc="lower right")
fig.tight_layout()
fig.savefig("outputs/myh7_rule_level_2026_comparison.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()


## Combined 4-panel figure (publication-ready)

Self-contained: reloads directly from the checkpoint files already saved above (`outputs/myh7_57_batch.jsonl`, `outputs/myh7_rule_audit.csv`, `outputs/myh7_expert_panel_current.csv`, and the hand-verified root-cause category dict), rather than depending on in-memory state from earlier cells -- same reasoning as the rest of this notebook's live-lookup cells. No panel titles by design (figure legend covers that in the write-up); panel letters (A-D) and axes/legends/value-labels are kept.

- **A** -- three classifiers, same 57 variants (raw 2018-snapshot comparison)
- **B** -- per-code discordance frequency vs. 2018 ClinGen
- **C** -- ClinGen (2026) vs. reimplementation (2026), n=46, fair contemporaneous comparison
- **D** -- root-cause breakdown of the 15 discordant calls in C

In [ ]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_TIER_ORDER = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.patch.set_facecolor("#fcfcfb")
_panel_labels = ["A", "B", "C", "D"]

def _label_panel(ax, letter):
    ax.text(-0.12, 1.03, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", color="#0b0b0b")

# ============================================================
# Panel A: three classifiers, same 57 variants
# ============================================================
ax = axes[0, 0]
_records = []
with open("outputs/myh7_57_batch.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            _records.append(json.loads(line))
myh7_results_df = pd.DataFrame(_records)
myh7_results_df = myh7_results_df[myh7_results_df.get("error").isna()] if "error" in myh7_results_df.columns else myh7_results_df

_SERIES = [
    ("ClinGen (2018)", "clingen_final", "#e1af64"),
    ("Original CardioClassifier (2018)", "original_cardioclassifier_final", "#86b6ef"),
    ("CardioClassifier reimplementation (Aug 2026)", "pipeline_classification", "#104281"),
]
_x = np.arange(len(_TIER_ORDER))
_width = 0.26
_a_max = 0
for i, (label, col, color) in enumerate(_SERIES):
    counts = myh7_results_df[col].value_counts().reindex(_TIER_ORDER, fill_value=0)
    _a_max = max(_a_max, counts.values.max())
    ax.bar(_x + (i - 1) * _width, counts.values, width=_width, color=color, label=label)
ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_ylim(0, _a_max * 1.35)  # headroom so the legend box doesn't overlap the tallest bars
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), frameon=False, fontsize=9)
_label_panel(ax, "A")

# ============================================================
# Panel B: per-code discordance frequency (missing/extra vs 2018 ClinGen)
# ============================================================
ax = axes[0, 1]
rule_audit_df = pd.read_csv("outputs/myh7_rule_audit.csv")

def _codes(s):
    if not isinstance(s, str) or s.strip() in ("-", ""):
        return []
    return [c.strip() for c in s.split(",") if c.strip() and c.strip() != "-"]

missing_counter, extra_counter = Counter(), Counter()
for _, row in rule_audit_df.iterrows():
    missing_counter.update(_codes(row["missing"]))
    extra_counter.update(_codes(row["extra"]))

_all_codes = sorted(set(missing_counter) | set(extra_counter), key=lambda c: -(missing_counter[c] + extra_counter[c]))
freq_df = pd.DataFrame({
    "code": _all_codes,
    "missing_in_n_variants": [missing_counter[c] for c in _all_codes],
    "extra_in_n_variants": [extra_counter[c] for c in _all_codes],
})
_y = np.arange(len(freq_df))
_h = 0.35
ax.barh(_y + _h/2, freq_df["missing_in_n_variants"], height=_h, color="#e87ba4", label="ClinGen (2018) fired, reimplementation didn't (missing)")
ax.barh(_y - _h/2, freq_df["extra_in_n_variants"], height=_h, color="#6da7ec", label="CardioClassifier reimplementation fired, ClinGen (2018) doesn't list (extra)")
ax.set_yticks(_y)
ax.set_yticklabels(freq_df["code"])
ax.invert_yaxis()
ax.set_xlabel(f"Number of variants (of {len(rule_audit_df)})", color="#52514e")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(frameon=False, fontsize=8, loc="lower right")
_label_panel(ax, "B")

# ============================================================
# Panel C: ClinGen (2026) vs CardioClassifier reimplementation (n=46)
# ============================================================
ax = axes[1, 0]
_current_df = pd.read_csv("outputs/myh7_expert_panel_current.csv")
_label_map = {"pathogenic": "Pathogenic", "likely pathogenic": "Likely Pathogenic",
              "uncertain significance": "VUS", "likely benign": "Likely Benign", "benign": "Benign"}

def _normalize_title(s):
    if not isinstance(s, str):
        return None
    key = s.strip().lower()
    return "VUS" if "uncertain" in key else _label_map.get(key, s.strip())

_current_df["current_clingen_tier"] = _current_df["expert_panel_current_significance"].map(_normalize_title)
_current_valid = _current_df.dropna(subset=["current_clingen_tier"]).copy()
_counts_clingen_now = _current_valid["current_clingen_tier"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_counts_original_cc_now = _current_valid["original_cc_final"].value_counts().reindex(_TIER_ORDER, fill_value=0)
_counts_pipeline_now = _current_valid["pipeline_final"].value_counts().reindex(_TIER_ORDER, fill_value=0)

_x = np.arange(len(_TIER_ORDER))
_width = 0.26
_c_max = max(_counts_clingen_now.values.max(), _counts_original_cc_now.values.max(), _counts_pipeline_now.values.max())
ax.bar(_x - _width, _counts_clingen_now.values, width=_width, color="#e1af64", label="ClinGen expert-panel assertion (retrieved Aug 2026)")
ax.bar(_x, _counts_original_cc_now.values, width=_width, color="#86b6ef", label="Original CardioClassifier (2018)")
ax.bar(_x + _width, _counts_pipeline_now.values, width=_width, color="#104281", label="CardioClassifier reimplementation (Aug 2026)")
ax.set_xticks(_x)
ax.set_xticklabels(_TIER_ORDER, rotation=15, ha="right")
ax.set_ylabel("Number of variants", color="#52514e")
ax.set_ylim(0, _c_max * 1.35)  # headroom so the legend box doesn't overlap the tallest bars
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), frameon=False, fontsize=9)
_label_panel(ax, "C")

print(f"Panel C: n={len(_current_valid)}, "
      f"v2 concordance={(_current_valid['current_clingen_tier']==_current_valid['pipeline_final']).mean():.1%}, "
      f"v1 concordance={(_current_valid['current_clingen_tier']==_current_valid['original_cc_final']).mean():.1%}")

# ============================================================
# Panel D: root cause of the 15 discordant calls (ClinGen 2026 vs reimplementation)
# ============================================================
ax = axes[1, 1]
_ps4_gap = {"c.2207T>C", "c.2513C>T", "c.2608C>T", "c.4258C>T", "c.5401G>A"}
_post2018_evidence = {"c.2539A>G", "c.4066G>A", "c.4130C>T", "c.3169G>A", "c.3578G>A", "c.5135G>A"}
_opp_boundary = {"c.4276G>A"}
_combined_case_data_generous = {"c.1157A>G"}
_bs1_related = {"c.5326A>G", "c.5329G>A"}

_CATEGORY_INFO = {
    "Evidence ClinGen incorporated\nafter 2018":                          (_post2018_evidence, "under"),
    "PS4: private case-cohort data\nunavailable to the reimplementation": (_ps4_gap, "under"),
    "BS1: outdated review or\nborderline population frequency":          (_bs1_related, "under"),
    "Reimplementation case data exceeds\nClinGen's cited evidence":       (_combined_case_data_generous, "over"),
    "Posterior-threshold correction\n(residual boundary case)":          (_opp_boundary, "over"),
}
_cat_counts = {label: len(cdnas) for label, (cdnas, _) in _CATEGORY_INFO.items()}
_cat_direction = {label: direction for label, (_, direction) in _CATEGORY_INFO.items()}
_ordered_labels = sorted(_cat_counts, key=lambda l: -_cat_counts[l])
_DIRECTION_COLORS = {"under": "#f08e67", "over": "#40a240"}
_colors = [_DIRECTION_COLORS[_cat_direction[l]] for l in _ordered_labels]
_values = [_cat_counts[l] for l in _ordered_labels]

_yy = range(len(_ordered_labels))
bars = ax.barh(_yy, _values, color=_colors, height=0.6)
for bar, v in zip(bars, _values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2, str(v), va="center", fontsize=10, color="#0b0b0b")
ax.set_yticks(_yy)
ax.set_yticklabels(_ordered_labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, max(_values) * 1.25)
ax.set_xlabel("Number of variants (of 15 discordant)", color="#52514e")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#f08e67", label="less pathogenic than ClinGen"),
    Patch(color="#40a240", label="more pathogenic than ClinGen"),
], loc="lower right", fontsize=8, frameon=False)
_label_panel(ax, "D")

fig.tight_layout()
fig.savefig("outputs/myh7_combined_4panel.png", dpi=300, facecolor=fig.get_facecolor())
print("saved outputs/myh7_combined_4panel.png")



## Combined 4-panel figure: clear vs. conflicting ClinVar variants (publication-ready)

Self-contained: reloads directly from the saved summary CSVs (`outputs/validation_clear_summary.csv` -- the ~2,500-variant clear/graded batch from `initial_validationn.ipynb`; `outputs/sandbox_conflicting_summary.csv` -- the 100-variant ClinVar-conflicting batch above), rather than depending on in-memory state from either notebook. No panel titles by design; panel letters (A-D) and axes/legends/value-labels are kept.

- **A** -- confusion matrix, pipeline vs. ClinVar, on variants where ClinVar has a confident (non-conflicting) classification
- **B** -- pipeline's continuous posterior probability, split by ClinVar Pathogenic/Benign ground truth
- **C** -- pipeline classification of the 100 ClinVar-*conflicting* variants (no agreed truth to score against)
- **D** -- which ACMG/AMP codes actually fire for the conflicting variants, split case-level (pink, all zero -- no case data supplied, by design) vs. computational (blue)

In [ ]:
import ast
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.patch.set_facecolor("#fcfcfb")

def _label_panel(ax, letter):
    ax.text(-0.12, 1.03, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", color="#0b0b0b")

_TIER_LABELS = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]

# ============================================================
# Shared source: the ~2,500-variant "clear/graded" batch (initial_validationn.ipynb)
# ============================================================
validation_results_df = pd.read_csv("outputs/validation_clear_summary.csv")

# ============================================================
# Panel A: confusion matrix, pipeline vs ClinVar (clear/graded variants)
# ============================================================
ax = axes[0, 0]
_CLINVAR_LABEL_MAP = {
    "benign": "Benign", "likely benign": "Likely Benign",
    "uncertain significance": "VUS", "vus": "VUS",
    "likely pathogenic": "Likely Pathogenic", "pathogenic": "Pathogenic",
}
cm_df = validation_results_df.copy()
cm_df["clinvar_tier_label"] = cm_df["clinvar_significance"].str.strip().str.lower().map(_CLINVAR_LABEL_MAP)
cm_df = cm_df.dropna(subset=["clinvar_tier_label"])

conf = pd.crosstab(cm_df["clinvar_tier_label"], cm_df["pipeline_classification"])
conf = conf.reindex(index=_TIER_LABELS, columns=_TIER_LABELS, fill_value=0)

log_conf = np.log1p(conf.values)
ax.imshow(log_conf, cmap="Blues", vmin=0, extent=(-0.5, 4.5, 4.5, -0.5), aspect="equal")
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(4.5, -0.5)
for i in range(5):
    for j in range(5):
        count = conf.values[i, j]
        if count == 0:
            continue
        rel = log_conf[i, j] / (log_conf.max() or 1)
        ax.text(j, i, str(count), ha="center", va="center", fontsize=11,
                 color="#ffffff" if rel > 0.6 else "#0b0b0b")
ax.set_xticks(range(5)); ax.set_xticklabels(_TIER_LABELS, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(_TIER_LABELS)
ax.set_xlabel("CardioClassifier reimplementation (Aug 2026)", color="#52514e")
ax.set_ylabel("ClinVar classification", color="#52514e")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks(np.arange(-0.5, 5, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 5, 1), minor=True)
ax.grid(which="minor", color="#fcfcfb", linewidth=2.5)
ax.tick_params(which="minor", length=0)
ax.tick_params(which="major", length=0)
_label_panel(ax, "A")

positive_clinvar = cm_df["clinvar_tier_label"].isin(["Pathogenic", "Likely Pathogenic"])
negative_clinvar = cm_df["clinvar_tier_label"].isin(["Benign", "Likely Benign"])
positive_pipeline = cm_df["pipeline_classification"].isin(["Pathogenic", "Likely Pathogenic"])
negative_pipeline = cm_df["pipeline_classification"].isin(["Benign", "Likely Benign"])
vus_pipeline = cm_df["pipeline_classification"] == "VUS"
TP = int((positive_clinvar & positive_pipeline).sum())
FN = int((positive_clinvar & (negative_pipeline | vus_pipeline)).sum())
TN = int((negative_clinvar & negative_pipeline).sum())
FP = int((negative_clinvar & positive_pipeline).sum())
# ClinVar-negative (Benign/Likely Benign) variants the pipeline called VUS. FN above
# already folds VUS-called *positives* in -- specificity's denominator must fold in
# VUS-called *negatives* the same way, or it's computed over a smaller, cleaner
# denominator than sensitivity and comes out inflated (was silently landing at 100%
# before this fix, since TN/(TN+FP) alone drops these 50 variants entirely).
FN_vus_neg = int((negative_clinvar & vus_pipeline).sum())
sensitivity = TP / (TP + FN) if (TP + FN) else float("nan")
specificity = TN / (TN + FP + FN_vus_neg) if (TN + FP + FN_vus_neg) else float("nan")
ppv = TP / (TP + FP) if (TP + FP) else float("nan")
npv = TN / (TN + FN) if (TN + FN) else float("nan")
print(f"Panel A: n={len(cm_df)}, TP={TP} FN={FN} TN={TN} FP={FP} FN_vus_neg={FN_vus_neg}, "
      f"sensitivity={sensitivity:.1%}, specificity={specificity:.1%}, PPV={ppv:.1%}, NPV={npv:.1%}")

# ============================================================
# Panel B: posterior probability distribution (clear/graded variants)
# ============================================================
ax = axes[0, 1]
hist_df = validation_results_df.copy()
hist_df["clinvar_group"] = hist_df["clinvar_significance"].str.strip().str.lower().map(
    lambda s: "Pathogenic" if "pathogenic" in s else ("Benign" if "benign" in s else None)
)
hist_df = hist_df.dropna(subset=["clinvar_group"])

_O_PP, _PRIOR = 2.08, 0.10
def _posterior_at(n_points):
    odds = _O_PP ** n_points
    return (odds * _PRIOR) / (odds * _PRIOR + (1 - _PRIOR))
_vus_lo, _vus_hi = _posterior_at(2), _posterior_at(4)
_band_spans = [(0.10, _vus_lo, "VUS-low (0-1 pts)"),
               (_vus_lo, _vus_hi, "VUS-mid (2-3 pts)"),
               (_vus_hi, 0.90, "VUS-high (4-5 pts)")]
for i, (lo, hi, _) in enumerate(_band_spans):
    ax.axvspan(lo, hi, color="#c1830f", alpha=0.06 + i * 0.06, zorder=0)

bins = np.linspace(0, 1, 41)
_hist_colors = {"Benign": "#6da7ec", "Pathogenic": "#e87ba4"}
for grp, color in _hist_colors.items():
    vals = hist_df.loc[hist_df["clinvar_group"] == grp, "posterior_probability"]
    ax.hist(vals, bins=bins, alpha=0.55, color=color, label=f"ClinVar {grp} (n={len(vals)})", edgecolor="none")
for x in (0.001, 0.10, 0.90, 0.99):
    ax.axvline(x, color="#b4b3ac", linewidth=1, linestyle="--", zorder=0)
for x in (_vus_lo, _vus_hi):
    ax.axvline(x, color="#c1830f", linewidth=0.8, linestyle=":", zorder=0)
ax.set_yscale("log")
ax.set_ylim(bottom=0.7)
for lo, hi, label in _band_spans:
    ax.text((lo + hi) / 2, -0.065, label, ha="center", va="top", fontsize=8,
             color="#8a5a06", transform=ax.get_xaxis_transform())
ax.set_xlabel("Pipeline posterior probability of pathogenicity", color="#52514e", labelpad=22)
ax.set_ylabel("Number of variants (log scale)", color="#52514e")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(loc="upper center", fontsize=9, frameon=False)
_label_panel(ax, "B")

# ============================================================
# Shared source: the 100-variant ClinVar-"conflicting" batch (benchmark_validation.ipynb)
# ============================================================
sandbox_results_df = pd.read_csv("outputs/sandbox_conflicting_summary.csv")

# ============================================================
# Panel C: pipeline classification of the conflicting variants
# ============================================================
ax = axes[1, 0]
_TIER_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
counts = sandbox_results_df["pipeline_classification"].value_counts().reindex(_TIER_LABELS, fill_value=0)
bars = ax.bar(_TIER_LABELS, counts.values, color=_TIER_COLORS, width=0.6)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.015,
            str(count), ha="center", va="bottom", fontsize=10, color="#0b0b0b")
ax.set_xticks(range(len(_TIER_LABELS)))
ax.set_xticklabels(_TIER_LABELS, rotation=15, ha="right")
ax.set_ylabel("Number of conflicting variants", color="#52514e")
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
_label_panel(ax, "C")

n = len(sandbox_results_df)
actionable = counts[["Pathogenic", "Likely Pathogenic", "Benign", "Likely Benign"]].sum()
print(f"Panel C: n={n}, {actionable}/{n} ({actionable/n:.0%}) resolved to actionable (P/LP/B/LB)")

# ============================================================
# Panel D: does the pipeline's call agree with what ClinVar submitters
# actually said? (replaces the ACMG rule-frequency panel)
# ============================================================
ax = axes[1, 1]
submitter_df = pd.read_csv("outputs/sandbox_conflicting_with_submitters.csv")
submitter_df["submitters_parsed"] = submitter_df["submitters"].apply(
    lambda s: json.loads(s) if isinstance(s, str) else []
)
agreement_base = submitter_df[submitter_df["submitters_parsed"].map(len) > 0].copy()

def _agreement_bucket(row):
    if row["matches_any_submitter"]:
        return "Matches a submitter"
    if row["within_submitter_range"]:
        return "Within submitted range"
    return "Outside submitted range"

agreement_base["agreement_bucket"] = agreement_base.apply(_agreement_bucket, axis=1)

_BUCKET_ORDER = ["Outside submitted range", "Within submitted range", "Matches a submitter"]
_BUCKET_COLORS = ["#86b6ef", "#6da7ec", "#5598e7"]

bucket_counts = agreement_base["agreement_bucket"].value_counts().reindex(_BUCKET_ORDER, fill_value=0)
n_agreement = len(agreement_base)

bars = ax.barh(_BUCKET_ORDER, bucket_counts.values, color=_BUCKET_COLORS, height=0.55, zorder=3)
_xmax_d = max(bucket_counts.values) if len(bucket_counts) and max(bucket_counts.values) > 0 else 1
for bar, count in zip(bars, bucket_counts.values):
    pct = count / n_agreement if n_agreement else 0
    ax.text(bar.get_width() + _xmax_d * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{count}  ({pct:.0%})", va="center", ha="left", fontsize=9, color="#0b0b0b")
ax.set_xlim(0, _xmax_d * 1.3)
ax.set_xlabel("Number of variants", color="#52514e")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
_label_panel(ax, "D")

n_excluded = len(submitter_df) - n_agreement
print(f"Panel D: n={n_agreement} conflicting variants with resolvable per-submitter data "
      f"({n_excluded} excluded, no resolvable submitter data)")

fig.tight_layout()
fig.savefig("outputs/clear_vs_conflicting_4panel.png", dpi=300, facecolor=fig.get_facecolor())
print("saved outputs/clear_vs_conflicting_4panel.png")
